# 第17章 选择、筛选与排序

使用loc、iloc、条件表达式、query和排序准确定位数据。


## 先解决一个小问题

拿一组小型业务数据练习“选择、筛选与排序”：先看数据结构，再完成一次明确的计算或转换。使用loc、iloc、条件表达式、query和排序准确定位数据。


## 这章为什么先学

这是“Pandas”路线中第 17 章的操作重点。本章只解决“选择、筛选与排序”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：按标签与位置选择


## 做完要留下什么

产出一个与“选择、筛选与排序”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 按标签与位置选择
- 组合多条件筛选
- 使用query表达条件
- 按一列或多列排序


## 核心概念

- loc按标签选择且切片包含终点，iloc按位置选择且右端不包含。
- 多个布尔条件必须分别加括号。
- 筛选前先检查缺失值和数据类型。


## 示例 1：loc与iloc

显式选择列可以减少无关数据进入后续计算。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东", "华北", "华南"],
    "channel": ["线上", "线下", "线上", "线上", "线下"],
    "amount": [320, 880, 460, 1250, 720],
}, index=["A1", "A2", "A3", "A4", "A5"])
print(orders.loc[["A2", "A4"], ["region", "amount"]])
print(orders.iloc[:3, [0, 2]])


## 示例 2：条件筛选与query

query适合可读的列条件，复杂动态逻辑仍可用布尔掩码。


In [ ]:
selected = orders[(orders["amount"] >= 500) & (orders["channel"] == "线上")]
queried = orders.query("amount >= 500 and region != '华北'")
print(selected)
print(queried)


## 示例 3：排序与Top N

稳定排序和明确方向有助于复现排名。


In [ ]:
ranked = orders.sort_values(["amount", "region"], ascending=[False, True])
top_three = orders.nlargest(3, "amount")
print(ranked)
print("Top 3:\n", top_three)


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd
from js import window

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = f"{window.location.origin}/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
completed = large_orders.query("status == '完成' and sales > 0")
high_value = completed.loc[
    completed["sales"] >= completed["sales"].quantile(0.99),
    ["order_id", "stock_code", "description", "country", "sales"]
].sort_values("sales", ascending=False)
print(f"Top 1% 高价值订单：{len(high_value):,} 条")
display(high_value.head(10))


## 常见误区

- loc和iloc切片边界规则混淆
- 多个条件之间漏写括号
- 排序后仍使用旧的位置含义


## 综合练习

1. 筛选华东或华南订单
2. 金额不低于400
3. 按金额降序返回前三条

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“筛选华东或华南订单”。
2. **独立完成**：不复制示例代码，完成“金额不低于400”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“按金额降序返回前三条”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华北", "华东", "西南"],
    "amount": [320, 880, 460, 1250, 720],
    "status": ["完成", "完成", "取消", "完成", "完成"],
})

# TODO: 筛选华东或华南且金额不低于400的订单
result = orders[
    orders["region"].isin(["华东", "华南"]) & (orders["amount"] >= 400)
]

# TODO: 按金额降序并返回前三条
result = result.sort_values("amount", ascending=False).head(3)

print(result)


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华北", "华东", "西南"],
    "amount": [320, 880, 460, 1250, 720],
    "status": ["完成", "完成", "取消", "完成", "完成"],
})
result = orders[
    orders["region"].isin(["华东", "华南"]) & (orders["amount"] >= 400)
].sort_values("amount", ascending=False).head(3)
print(result)

# 自检
assert len(result) == 3, "检查结果行数：应该返回3条记录"
assert result.iloc[0]["amount"] == 1250, "检查第一条：金额最大应该是1250"


## 本章小结

使用loc、iloc、条件表达式、query和排序准确定位数据。

**迁移思考**：

1. 如果需要筛选"金额大于500或地区为华东"的订单，布尔表达式应该如何写？
2. 为什么 loc 切片包含终点而 iloc 切片不包含？这种差异在什么场景下容易出错？


### 你已经掌握

- 按标签与位置选择
- 组合多条件筛选
- 使用query表达条件
- 按一列或多列排序


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| loc与iloc | 显式选择列可以减少无关数据进入后续计算。 | `pd.DataFrame()`、`loc[["A2", "A4"]`、`iloc[:3, [0, 2]` |
| 条件筛选与query | query适合可读的列条件，复杂动态逻辑仍可用布尔掩码。 | `orders.query()`、`orders[(orders["amount"]`、`orders["channel"]` |
| 排序与Top N | 稳定排序和明确方向有助于复现排名。 | `orders.sort_values()`、`orders.nlargest()` |


### 需要注意

- loc和iloc切片边界规则混淆
- 多个条件之间漏写括号
- 排序后仍使用旧的位置含义


### 完成检查

- [ ] 能够按标签与位置选择
- [ ] 能够组合多条件筛选
- [ ] 能够使用query表达条件
- [ ] 能够按一列或多列排序


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
